# Phase 2 - Exploratory Data Analysis: NASA C-MAPSS FD001

**Scope.** Characterize the FD001 training fleet: engine lifetimes, sensor behaviour, degradation trends and data quality. This notebook uses the project package functions (`rul_prediction.data.loader`, `rul_prediction.data.validation`) and does not duplicate loader code.

**Leakage guardrails in force.** Training data only. The official test set (and its RUL labels) is not used in this analysis. No sensor is removed - constant-column detection is reported for later ablation decisions (Phase 8). RUL is computed here for EDA only; the production preprocessing pipeline (Phase 4) reintroduces it separately.

## 0. Imports and package versioning

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

# Anchor to the repository root so relative data paths resolve regardless of
# where the kernel is launched from (e.g. nbconvert runs with cwd = notebooks/).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data' / 'raw').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print('anchored project root: <repo root, detected at runtime>')

from rul_prediction import __version__
from rul_prediction.data.loader import SENSOR_COLUMNS, SETTING_COLUMNS, load_train, summarize
from rul_prediction.data.validation import validate_frame

print('rul_prediction', __version__)
print('pandas', pd.__version__, '| numpy', np.__version__, '| matplotlib', matplotlib.__version__)

FIG_DIR = Path('reports/figures/eda')
FIG_DIR.mkdir(parents=True, exist_ok=True)

anchored project root: <repo root, detected at runtime>
rul_prediction 0.1.0
pandas 3.0.5 | numpy 2.5.2 | matplotlib 3.11.1


## 1. Data load and integrity re-check

In [2]:
train = load_train('FD001')
report = validate_frame(train, 'FD001', 'train')
assert report.passed, 'FD001 training data failed integrity validation'
print('shape:', train.shape)
print('engines:', int(train['engine_id'].nunique()))
print('constant columns (reported, kept):', report.diagnostics['constant_columns'])

shape: (20631, 26)
engines: 100
constant columns (reported, kept): ['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


## 2. Engine lifetime distribution

Lifetime = maximum cycle reached by each training engine.

In [3]:
summary = summarize(train, 'FD001', 'train')
for key in ('min_lifetime', 'max_lifetime', 'mean_lifetime', 'median_lifetime', 'std_lifetime'):
    print(key, '=', summary[key])
lifetime = train.groupby('engine_id')['cycle'].max()

min_lifetime = 128
max_lifetime = 362
mean_lifetime = 206.31
median_lifetime = 199.0
std_lifetime = 46.3427492067573


In [4]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(lifetime, bins=15, edgecolor='black', alpha=0.85)
ax.set_xlabel('Lifetime (cycles to failure)')
ax.set_ylabel('Number of engines')
ax.set_title('FD001 training engine lifetime distribution')
fig.tight_layout()
fig.savefig(FIG_DIR / 'lifetime_distribution.png', dpi=150)
plt.close(fig)
print('saved:', FIG_DIR / 'lifetime_distribution.png')

saved: reports\figures\eda\lifetime_distribution.png


## 3. Sensor variance

Per-sensor standard deviation over all training cycles. Near-constant sensors cannot carry degradation information; high-variance sensors are candidates for degradation-sensitive features.

In [5]:
sensor_std = train[SENSOR_COLUMNS].std().sort_values(ascending=False)
print(sensor_std.round(3).to_string())
print('\nTop-5 variance:', list(sensor_std.head(5).index))
print('Zero variance:', list(sensor_std[sensor_std == 0].index))
print('Effectively constant (<1e-12):',
      list(sensor_std[(sensor_std > 0) & (sensor_std < 1e-12)].index))

sensor_9     22.083
sensor_14    19.076
sensor_4      9.001
sensor_3      6.131
sensor_17     1.549
sensor_7      0.885
sensor_12     0.738
sensor_2      0.500
sensor_11     0.267
sensor_20     0.181
sensor_21     0.108
sensor_13     0.072
sensor_8      0.071
sensor_15     0.038
sensor_6      0.001
sensor_5      0.000
sensor_16     0.000
sensor_1      0.000
sensor_10     0.000
sensor_19     0.000
sensor_18     0.000

Top-5 variance: ['sensor_9', 'sensor_14', 'sensor_4', 'sensor_3', 'sensor_17']
Zero variance: ['sensor_1', 'sensor_10', 'sensor_19', 'sensor_18']
Effectively constant (<1e-12): ['sensor_5', 'sensor_16']


In [6]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(sensor_std.index, sensor_std.values)
ax.set_xticks(range(len(sensor_std)))
ax.set_xticklabels(sensor_std.index, rotation=90)
ax.set_ylabel('Standard deviation (raw units)')
ax.set_title('FD001 per-sensor standard deviation (training)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'sensor_variance.png', dpi=150)
plt.close(fig)
print('saved:', FIG_DIR / 'sensor_variance.png')

saved: reports\figures\eda\sensor_variance.png


## 4. Sensor trajectories

Top-variance sensors, plotted against cycle for three representative engines: shortest lifetime, median lifetime, longest lifetime.

In [7]:
lifetime_sorted = lifetime.sort_values()
engines = [lifetime_sorted.index[0],
           lifetime_sorted.index[len(lifetime_sorted) // 2],
           lifetime_sorted.index[-1]]
print('short / mid / long lifetime engines:', engines,
      '->', [int(lifetime_sorted.loc[e]) for e in engines], 'cycles')

top_sensors = list(sensor_std.head(5).index)

short / mid / long lifetime engines: [np.int64(39), np.int64(79), np.int64(69)] -> [128, 199, 362] cycles


In [8]:
fig, axes = plt.subplots(len(top_sensors), 1, figsize=(10, 2.3 * len(top_sensors)), sharex=True)
for ax, s in zip(axes, top_sensors):
    for e in engines:
        sub = train[train['engine_id'] == e]
        ax.plot(sub['cycle'], sub[s], label='engine %d (%d cycles)' % (e, len(sub)))
    ax.set_ylabel(s)
axes[-1].set_xlabel('Cycle')
axes[0].legend(ncol=3, fontsize=8)
fig.suptitle('Top-variance sensor trajectories (short / mid / long-lived engines)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'sensor_trajectories.png', dpi=150)
plt.close(fig)
print('saved:', FIG_DIR / 'sensor_trajectories.png')

saved: reports\figures\eda\sensor_trajectories.png


## 5. RUL construction and relationship to sensors (training only)

Training RUL is defined per engine as `max_cycle(engine) - current_cycle`. This is the standard C-MAPSS run-to-failure definition and is used for EDA only.

In [9]:
train = train.copy()
train['rul'] = train.groupby('engine_id')['cycle'].transform('max') - train['cycle']
print(train[['engine_id', 'cycle', 'rul']].head(3).to_string(index=False))
print('\nRUL describe:')
print(train['rul'].describe().round(2).to_string())

 engine_id  cycle  rul
         1      1  191
         1      2  190
         1      3  189

RUL describe:
count    20631.00
mean       107.81
std         68.88
min          0.00
25%         51.00
50%        103.00
75%        155.00
max        361.00


## 6. Sensor to RUL correlation

Pearson correlation between each non-constant sensor and RUL, computed on training rows. Sensors whose values drift monotonically with degradation should show high |correlation| with RUL.

In [10]:
varying = [c for c in SENSOR_COLUMNS if train[c].std() > 1e-12]
sensor_rul_corr = train[varying].corrwith(train['rul'])
print(sensor_rul_corr.abs().sort_values(ascending=False).round(3).to_string())
print('\nconstant sensors excluded:', len(SENSOR_COLUMNS) - len(varying))

sensor_11    0.696
sensor_4     0.679
sensor_12    0.672
sensor_7     0.657
sensor_15    0.643
sensor_21    0.636
sensor_20    0.629
sensor_2     0.606
sensor_17    0.606
sensor_3     0.585
sensor_8     0.564
sensor_13    0.563
sensor_9     0.390
sensor_14    0.307
sensor_6     0.128

constant sensors excluded: 6


In [11]:
fig, ax = plt.subplots(figsize=(7, 5.5))
order = sensor_rul_corr.abs().sort_values(ascending=False)
ax.barh(order.index, sensor_rul_corr.loc[order.index].values)
ax.set_xlabel('Pearson correlation with RUL')
ax.set_title('FD001 sensor-RUL correlation (training)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'sensor_rul_correlation.png', dpi=150)
plt.close(fig)
print('saved:', FIG_DIR / 'sensor_rul_correlation.png')

saved: reports\figures\eda\sensor_rul_correlation.png


## 7. Sensor-sensor correlations

Heatmap among the five sensors most correlated with RUL.

In [12]:
top5 = list(sensor_rul_corr.abs().sort_values(ascending=False).head(5).index)
print('top-5 RUL-correlated sensors:', top5)
sub_corr = train[top5].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sub_corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(top5)))
ax.set_xticklabels(top5, rotation=45, ha='right')
ax.set_yticks(range(len(top5)))
ax.set_yticklabels(top5)
for i in range(len(top5)):
    for j in range(len(top5)):
        ax.text(j, i, '%.2f' % sub_corr.values[i, j], ha='center', va='center', fontsize=8)
ax.set_title('Sensor-sensor correlation (top RUL sensors)')
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'sensor_sensor_heatmap.png', dpi=150)
plt.close(fig)
print('saved:', FIG_DIR / 'sensor_sensor_heatmap.png')

top-5 RUL-correlated sensors: ['sensor_11', 'sensor_4', 'sensor_12', 'sensor_7', 'sensor_15']


saved: reports\figures\eda\sensor_sensor_heatmap.png


## 8. Operating settings in FD001

FD001 is a single-operating-condition environment (sea level). The settings should therefore be (nearly) constant.

In [13]:
print('setting std (training):')
print(train[SETTING_COLUMNS].std().round(6).to_string())
print('unique values:', train[SETTING_COLUMNS].nunique().to_dict())

setting std (training):
setting_1    0.002187
setting_2    0.000293
setting_3    0.000000
unique values: {'setting_1': 158, 'setting_2': 13, 'setting_3': 1}


## 9. Data quality observations

Integrity validation (cell 1) passed for FD001 training data: 26 expected columns, all numeric, no missing values, no infinite values, no duplicate `(engine_id, cycle)` records, cycles strictly increasing within every engine, 100 engines with contiguous IDs.

One structural observation: 7 of 26 columns are constant (or effectively constant) across the whole training set - `setting_3`, `sensor_1`, `sensor_5`, `sensor_10`, `sensor_16`, `sensor_18`, `sensor_19` (measured standard deviation 0 or below 1e-12). They are intentionally retained; whether to drop them is an ablation question for Phase 8.

## 10. Evidence-based conclusions

1. **Lifetime variability is high.** Training lifetimes range from **128 to 362 cycles** (mean 206.31, median 199, std 46.34). Short and long-lived engines differ by a factor of ~2.8, so degradation rates are heterogeneous; sequence-based models must still see both populations.
2. **Seven columns are signal-free in FD001.** `setting_3`, `sensor_1`, `sensor_5`, `sensor_10`, `sensor_16`, `sensor_18`, `sensor_19` have (effectively) zero variance and cannot contribute degradation information. They are candidates for the Phase 8 sensor-ablation experiment, not removed here.
3. **Degradation candidates exist.** Highest-variance sensors: `sensor_9` (std 22.08), `sensor_14` (19.08), `sensor_4` (9.00), `sensor_3` (6.13). Trajectories show clear drift over cycles for these sensors on all three representative engines.
4. **RUL is strongly linearly tracked by several sensors.** Highest |Pearson correlation| with RUL: `sensor_11` (0.70), `sensor_4` (0.68), `sensor_12` (0.67), `sensor_7` (0.66), `sensor_15` (0.64). These are prime inputs for both classical and sequence models.
5. **Operating settings carry negligible information in FD001.** `setting_3` is exactly constant; `setting_1` and `setting_2` vary by std 0.0022 / 0.0003 only and correlate with RUL at ~0. This is expected for a single-condition dataset and justifies treating FD001 as condition-homogeneous (a contrast with FD004 in Phase 15).
6. **No obvious data-quality issues.** All integrity checks passed; no missing/infinite values, no duplicate or unordered cycle records.

**Implication for later phases.** Preprocessing should scale the ~14 varying sensors (constant columns add nothing but can remain to keep the pipeline conservative until Phase 8 tests removal); sequence windows of length 30 are compatible with the shortest lifetime of 128 cycles.